RAG: Retrieval Augmented Genertion.

Why this exists:
* Large Language Models(LLMs) have hard boundary: they know only what theu were trained on, they are prone to making things up (hallucinating) when they are pushed past that data.

* So, instead

RAG Architecture: Pipeline:

Phase A: Ingestion
1. Documents
2. Chunks
3. Embed
4. Store

Phase B: Inference
1. User Qn
2. Embed Qn
3.

**RAG SYSTEM**

In [ ]:
!pip install sentence-transformers chromadb groq pandas -q

# sentence-transformers - string to integer conversion
# chromadb              - our vector

In [ ]:
import pandas as pd
# loading and managing dataset

import chromadb
# the vector db library to store document embeddings and perform similarity search

from sentence_transformers import SentenceTransformer
# class that loads pretrained LLMs

from groq import Groq
# Groq: The Groq API client class for calling LLMs

import os
# os:

from chromadb.utils import embedding_functions
#

In [ ]:
GROQ_API_KEY = "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"

# this makes it accessible to GROQ library when it needs to authenticate
os.environ["GROQ_API_KEY"]=GROQ_API_KEY

groq_client =  Groq(api_key=GROQ_API_KEY)

print("Groq API client installed.")
print("Note: If you see an authentication error later, double check your API key")

In [ ]:
df=pd.read_csv('college_notes.csv')

print("Shape of dataset:", {df.shape})

print("Column names:", df.columns.tolist())

print("First 3 rows:")
print(df.head(3))

In [ ]:
print("Subjects in the dataset:")
print(df["subject"].value_counts())

print('Sample of topics:')
print(df[['note_id', 'subject', 'topic']].to_string(index=False))

print('Length of content (number of characters) for each notes')
df['content_length']=df['content'].apply(len)
print(df[['topic', 'content_length']].to_string(index=False))

In [ ]:
document = df['content'].tolist()

ids = [f"note_{row['note_id']}" for row in df.to_dict('records')]

metadatas = [
    {"subject": row['subject'], "topic": row['topic']}
    for row in df.to_dict('records')
]
print(f"Total chunks prepared: {len(document)}")
print(f"Fist document ID: {ids[0]}")
print(f"First document metadata: {metadatas[0]}")
print(f"First document content: {document[0][:100]}...")

In [ ]:
print('Loading embedding model...')
print("This may take 30-60 seconds on first run - model is being downloaded")
print("Subsequent run will be faster as the model is cached")

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print("\nEmbedding model loaded successfully")

test_embedding = embedding_model.encode("This is a test sentence")
print(f"Test embedding shape: {test_embedding.shape}")
print(f"First 5 values of test embedding: {test_embedding[:5]}")

In [ ]:
chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(name="college_notes_rag")
print("chromaDB client created")
print("Collection name: college_notes_rag")
print(f"Documents in collection so far: {collection.count()}")

In [ ]:
print("Generating embedding for all 15 notes")
embeddings=embedding_model.encode(document, show_progress_bar=True)
print(f"\nEmbedding matrix shape: {embeddings.shape}")
# print(f"First 5 values of first embedding: {embeddings[0][:5]}")

embeddings_list=embeddings.tolist()
collection.add(
    documents=document,
    embeddings=embeddings_list,
    metadatas=metadatas,
    ids=ids
)
print("\nDocuments successfully addded to chromaDB")
print(f"\nTotal documents in collection: {collection.count()}")

In [ ]:
def retrieve_relevant_chunks(question, top_k=3):
  """
  Given a user question, retrieve the most relevant document chunks form ChromaDB.

  Parameters:
      question (str) : The user's questions as a text string
      top_k     (int) : How many top results to return (default: 3)

  Returns:
      A dictionary containing the retrived documents, distances and their metadata
  """
  question_embedding = embedding_model.encode(question).tolist()
  results = collection.query(
      query_embeddings=[question_embedding],
      n_results=top_k
  )

  return results

print("Retrival function defined successfully!")
print("Function: retrieve_relevant_chunks(question, top_k=3)")

In [ ]:
test_question="What is ETL"
results = retrieve_relevant_chunks(test_question, top_k=3)
print("\nTop 3 Retrieved Chunks:")
print('-'*30)

In [ ]:
for i, (doc, dist, meta) in enumerate(zip(results['documents'][0], results['distances'][0], results['metadatas'][0])):
  print(f"Document {i+1}")
  print(f"\nResult {i+1}:")
  print(f"  Subject   : {meta['subject']}")
  print(f"  Topic    : {meta['topic']}")
  print(f"  Distance : {dist}")
  print(f"  Content  : {doc[:100]}...")

In [ ]:
def build_context_from_results(results):
  """
  Format ChromaDB retrieval results into a relatable context string.

  Parameters:
      results: The output from collection.query() - a dictionary

  Returns:
      context_str (str) : A formatted string of all retrieved document chunks.
  """

  context_parts = []

  for i, (doc, meta) in enumerate(zip(
      results['documents'][0],
      results['metadatas'][0]
  )):
    context_text = f"[Source {i+1}: {meta['subject']} - {meta['topic']}\n{doc}]"
    context_parts.append(context_text)

    return context_parts

In [ ]:
context_built = build_context_from_results(results=results)
for i, context in enumerate(context_built):
  print(f"Context {i+1}:\n{context}\n")

In [ ]:
# BUILD RAG Generation function
# This function sends the context + question to Groq LLM and returns the answer

def generate_rag_answer(question, context):
  """
  Send the restricted context + question to Groq LLM for answer genration.

  Parameters:
    question (str)    :  The user's question
    context  (str)    :  The restricted context chunks(formatted string)

  Returns:
    answer  (str)     :   The LLM's generated answer
  """

  # SYSTEM PROMPT: Instructions to the LLM about its role and behaviour
  # This is the key to RAG - we tell the LLM to ONLY use the provided context

  system_prompt = """ You are a helpful academic assistant for engineering students.

  You will be given an context retrieved from a college knowledge base, and a student's question.

  RULES:
  1. Answer only using the instruction provided in teh context below
  2. If the answer is not found in the context, say exactly
     "I don't have enough information in my knowledge base to answer this question"
  3. Do not use you general training knowledge
  4. Keep answer clear, accurate, and begineeer friendly
  5. Mention which source the information came from when possible."""

  # USER PROMPT: the context + quesion formatted

  user_prompt=f"""Context"
  {context}

  Question:
  {question}
  Please answer the question using only on the context provided above."""

  response = groq_client.chat.completions.create(
      model="gpt-3.5-turbo", # llama-3.1.8b-instant-to use same model from traning
      messages=[
          {"role":"system","content":system_prompt},
          {"role":"user","content":user}
      ],
      temperature = 0.1,
#temperature - 0.1 - Very low randomness - we want factual, consistent answers, for RAG, low temp is preffered so the LLM sticks to the context
      max_tokens = 500 #max len of gen responce
  )

  #Extract the text answer from the API resonse object
  answer = response.choices[0].message.content
  #response.choices : A list of responses
  return answer
print("Defined RAG generation function")

In [ ]:
def ask_college_assistant(question, top_k=3, verbose=True):
    """
    Parameters:
      question (str) : The user's question
      top_k    (int): Returns the top 3 chunks from the collection
      verbose (bool): Whether to print intermediate steps (True)

    Returns:
      answer (str): The final generated answer
    """
    if verbose:
      print(f"Question: {question}")
      print("="*60)
      print("Step 1: Retrieving Relevant Chunks...")
      print("="*60)

    results = retrieve_relevent_chunks(question, top_k=top_k)

    if verbose:
      print("Step 2: Building Context...")
      print("="*60)
    context = build_context_from_results(results)

    if verbose:
      print("Step 3: Generating Answer...")
      print("="*60)
    answer = generate_rag_answer(question, context)

    return answer